# TSR Claude và GA memetic trên LR108

Chọn runtime **CPU** rồi chạy ô bên dưới. Ô này chạy đúng hai phương pháp trên `lr108-5-uniform.csv` với 50 tác vụ và 10 robot. Cả hai dùng cùng nghiệm greedy ban đầu và cùng 5 seed. Giới hạn mặc định là 600 giây CPU cho mỗi phương pháp ở mỗi seed.

TSR Claude dùng cấu hình benchmark chung với ngưỡng restart 300/600. GA memetic dùng bốn lần thử lai, đột biến đổi thứ tự, tỷ lệ đột biến thích ứng và hai bước cải thiện cục bộ theo Action Plan. Nghiệm greedy chung là cá thể đầu tiên trong quần thể GA.

In [ ]:
CPU_SECONDS = 600
SEEDS = [0, 1, 2, 3, 4]
WORKERS = 5

from hashlib import sha256
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

from IPython.display import display

REPO = "https://github.com/dhducpro/colab_test.git"
REVISION = "33c0ab7f2d07226a315358b4681712c7e850bd8d"
workspace = Path(tempfile.mkdtemp(prefix="pdpcr_pair_"))
source = workspace / "colab_test"
subprocess.run(["git", "clone", "--quiet", REPO, str(source)], check=True)
subprocess.run(["git", "-C", str(source), "checkout", "--quiet", "--detach", REVISION], check=True)

env = dict(os.environ, PYTHONHASHSEED="0", PYTHONIOENCODING="utf-8")
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    env[name] = "1"

probe = "import numpy, pandas, networkx, scipy, dateutil; assert int(dateutil.__version__.split('.')[0]) < 3"
if subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True).returncode:
    dependencies = workspace / "dependencies"
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(dependencies),
                    "-r", str(source / "requirements.txt")], env=env, check=True)
    env["PYTHONPATH"] = os.pathsep.join(filter(None, (str(dependencies), env.get("PYTHONPATH"))))
subprocess.run([sys.executable, "-c", probe], env=env, check=True)

settings = json.dumps([REVISION, float(CPU_SECONDS), SEEDS, WORKERS])
run_id = sha256(settings.encode()).hexdigest()[:12]
out = Path.cwd() / "pdpcr_results" / f"claude_ga_{run_id}"
subprocess.run(
    [sys.executable, "-u", str(source / "run_hung_study.py"),
     "--instances", str(source / "data/lr108-5-uniform.csv"),
     "--methods", "claude", "ga_pilot",
     "--seconds", str(CPU_SECONDS), "--seeds", *map(str, SEEDS),
     "--workers", str(WORKERS), "--out", str(out)],
    env=env, check=True,
)

import pandas as pd
from shutil import make_archive

labels = {"claude": "TSR Claude", "ga_pilot": "GA memetic"}
runs = pd.read_csv(out / "runs.csv")
print(f"Makespan greedy chung: {runs['initial'].iloc[0]:.6f}")
summary = pd.read_csv(out / "summary.csv").replace({"method": labels})
display(summary[["method", "seeds", "mean", "sd", "best"]].round(4))
per_seed = runs.pivot(index="seed", columns="method", values="makespan").rename(columns=labels)
display(per_seed.round(4))

archive = make_archive(str(out), "zip", root_dir=out)
try:
    from google.colab import files
except ImportError:
    print(archive)
else:
    files.download(archive)